# Monte Carlo production & inventory planner (multi‑product)

This notebook builds a 10‑year Monte Carlo model for seed production and inventory,
using archetype/maturity inputs, yield & conversion uncertainty, and portfolio‑level
aggregation across multiple products.


**Imports and file paths**

This block imports core Python and Plotly libraries, and defines paths to both
data files:

- `data.xlsx` — conversion rates, production yields, product parameters, and
  median sales (used as fallback)
- `salesVariability.xlsx` — lognormal parameters (mu and sigma²) for first-year
  sales and year-over-year growth rates, fitted from historical data at the
  archetype/maturity level.  These replace the fixed median sales curve in the
  Monte Carlo loop so that each simulation run draws its own unique sales
  trajectory.


In [13]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display

# Plotly renderer (adjust if using local Jupyter)
pio.renderers.default = "colab"

# Path to main Excel file (conversion rates, yields, product parameters, median sales)
file_path = "/content/data.xlsx"           # change if needed

# Path to sales variability file (lognormal parameters for first-year sales and growth rates)
sales_var_path = "/content/salesVariability.xlsx"   # change if needed


**Load sheets and basic cleaning**

This block loads the key Excel sheets (conversion rates, production yields, product parameters, sales volume parameters), standardizes their column names, and builds a clean Parent0→Archetype mapping used to join other datasets later.



In [14]:
# Load sheets
conv_tab   = pd.read_excel(file_path, sheet_name="Conversion rates")
yield_tab  = pd.read_excel(file_path, sheet_name="Production yields")
params_tab = pd.read_excel(file_path, sheet_name="Product parameters")
sales_raw  = pd.read_excel(file_path, sheet_name="Sales volume parameters", header=None)

# Clean column names
for df_ in [conv_tab, yield_tab, params_tab]:
    df_.columns = df_.columns.astype(str).str.strip()

# Validate and normalize Parent0 → Archetype mapping
for col in ["Parent0", "Archetype"]:
    if col not in params_tab.columns:
        raise ValueError("Product parameters must contain columns: Parent0, Archetype")

params_tab["Parent0"]  = params_tab["Parent0"].astype(str).str.strip()
params_tab["Archetype"] = params_tab["Archetype"].astype(str).str.strip()
parent_to_arch = params_tab[["Parent0", "Archetype"]].dropna()


**Sales variability parameters (lognormal)**

This block loads `salesVariability.xlsx` and parses four parameter tables used
to stochastically generate sales curves inside the Monte Carlo loop:

- **First-year sales mu** — log-scale mean of Year 1 sales by archetype/maturity
- **First-year sales sigma²** — log-scale variance of Year 1 sales
- **Growth rate mu** — log-scale mean of YoY growth for each year (Years 2-10)
- **Growth rate sigma²** — log-scale variance of YoY growth for each year

During simulation, each run draws Year 1 sales and each year's growth rate
independently from `Lognormal(mu, sigma²)`, producing a unique 10-year sales
trajectory per run rather than a single fixed median curve.

Values of `-9.2103` in the growth rate tables represent `log(~0)`, signalling
end-of-lifecycle for that archetype/maturity — sales naturally collapse to near
zero without needing a hard cutoff.


In [15]:
# ── Load salesVariability.xlsx ───────────────────────────────────────────────
sv_raw = pd.read_excel(sales_var_path,
                       sheet_name="Sales volume parameters",
                       header=None)

# Helper: find a value in the raw sheet
def sv_find(text):
    t = text.strip().lower()
    for r in range(sv_raw.shape[0]):
        for c in range(sv_raw.shape[1]):
            v = sv_raw.iat[r, c]
            if not pd.isna(v) and t in str(v).strip().lower():
                return r, c
    return None

def sv_scan_cols(r, start_c):
    """Return end column index of a contiguous non-blank run."""
    end = start_c
    while end < sv_raw.shape[1]:
        v = sv_raw.iat[r, end]
        if pd.isna(v) or str(v).strip() == "":
            break
        end += 1
    return end

def parse_sv_wide(header_row, start_col, mat_cols, label):
    """
    Parse a wide block with columns: Archetype | mat1 | mat2 | ...
    Returns dict keyed by (archetype, maturity) -> float
    """
    result = {}
    r = header_row + 1
    while r < sv_raw.shape[0]:
        arch = sv_raw.iat[r, start_col]
        if pd.isna(arch) or str(arch).strip() == "":
            break
        arch = str(arch).strip()
        for mat, col in mat_cols.items():
            val = sv_raw.iat[r, col]
            if not pd.isna(val):
                result[(arch, int(mat))] = float(val)
        r += 1
    return result

def parse_sv_long(header_row, arch_col, year_cols, label):
    """
    Parse a long block with columns: Archetype | Maturity | yr2 | yr3 | ...
    Returns dict keyed by (archetype, maturity, year) -> float
    """
    result = {}
    r = header_row + 1
    while r < sv_raw.shape[0]:
        arch = sv_raw.iat[r, arch_col]
        mat  = sv_raw.iat[r, arch_col + 1]
        if pd.isna(arch) or str(arch).strip() == "":
            break
        arch = str(arch).strip()
        if pd.isna(mat):
            r += 1
            continue
        mat = int(float(mat))
        for yr, col in year_cols.items():
            val = sv_raw.iat[r, col]
            if not pd.isna(val):
                result[(arch, mat, int(yr))] = float(val)
        r += 1
    return result

# ── First-year sales mu (wide block, rows 2-9, cols 0-4) ──
fy_mu_header = 4   # row containing "Archetype" header for first block
fy_mat_cols  = {85: 1, 95: 2, 105: 3, 115: 4}
sv_fy_mu = parse_sv_wide(fy_mu_header, 0, fy_mat_cols, "first-year mu")

# ── First-year sales sigma² (wide block, rows 12-19, cols 0-4) ──
sig2_anchor = sv_find("Log-normal (sigma^2)")
if sig2_anchor is None:
    raise ValueError("Could not find 'Log-normal (sigma^2)' block in salesVariability.xlsx")
sig2_header = sig2_anchor[0] + 2   # skip label row + maturity row to get to Archetype header
sv_fy_sig2  = parse_sv_wide(sig2_header, 0, fy_mat_cols, "first-year sigma2")

# ── Growth rate mu (long block, cols 7-22) ──
# Header row 4: Archetype(7) | Maturity(8) | 2(9) | 3(10) | ... | 10(16)
gr_mu_arch_col = 7
gr_mu_year_cols = {yr: 9 + (yr - 2) for yr in range(2, 11)}   # yr2=col9, yr3=col10, ...yr10=col16
sv_gr_mu = parse_sv_long(4, gr_mu_arch_col, gr_mu_year_cols, "growth mu")

# ── Growth rate sigma² (long block, cols 24-39) ──
# Header row 4: Archetype(24) | Maturity(25) | 2(26) | 3(27) | ... | 10(32)
gr_sig2_arch_col = 24
gr_sig2_year_cols = {yr: 26 + (yr - 2) for yr in range(2, 11)}
sv_gr_sig2 = parse_sv_long(4, gr_sig2_arch_col, gr_sig2_year_cols, "growth sigma2")

# ── Validate coverage ──
archetypes_sv = sorted(set(k[0] for k in sv_fy_mu.keys()))
maturities_sv = sorted(set(k[1] for k in sv_fy_mu.keys()))
print(f"Sales variability loaded.")
print(f"  Archetypes : {archetypes_sv}")
print(f"  Maturities : {maturities_sv}")
print(f"  First-year mu entries   : {len(sv_fy_mu)}")
print(f"  First-year sig2 entries : {len(sv_fy_sig2)}")
print(f"  Growth mu entries       : {len(sv_gr_mu)}")
print(f"  Growth sig2 entries     : {len(sv_gr_sig2)}")


Sales variability loaded.
  Archetypes : ['Bayer - Above ground', 'Bayer - Above/below ground', 'Conventional', 'Syngenta - above ground', 'Syngenta - above/below ground']
  Maturities : [85, 95, 105, 115]
  First-year mu entries   : 20
  First-year sig2 entries : 20
  Growth mu entries       : 180
  Growth sig2 entries     : 180


**Yield and conversion prep**

This block prepares agronomic and processing uncertainty inputs by computing yield factors (actual vs planned), cleaning numeric fields, and joining yields and conversion rates to archetypes so we can later sample realistic yield and conversion variability by product.



In [16]:
# Yield: compute Yield_Factor = Actual / Planned
for col in ["Parent0", "Planned yield (bu/ac)", "Actual yield"]:
    if col not in yield_tab.columns:
        raise ValueError(f"Production yields sheet missing column: {col}")

yield_tab["Parent0"] = yield_tab["Parent0"].astype(str).str.strip()
yield_tab["Planned yield (bu/ac)"] = pd.to_numeric(yield_tab["Planned yield (bu/ac)"], errors="coerce")
yield_tab["Actual yield"]          = pd.to_numeric(yield_tab["Actual yield"], errors="coerce")

yield_tab["Yield_Factor"] = yield_tab["Actual yield"] / yield_tab["Planned yield (bu/ac)"]
yield_tab["Yield_Factor"] = yield_tab["Yield_Factor"].replace([np.inf, -np.inf], np.nan)

# Map archetype onto yield rows
yield_w_arch = yield_tab.merge(parent_to_arch, on="Parent0", how="left")

# Conversion table
for col in ["Parent0", "totalConversionRate"]:
    if col not in conv_tab.columns:
        raise ValueError(f"Conversion rates sheet missing column: {col}")

conv_tab["Parent0"] = conv_tab["Parent0"].astype(str).str.strip()
conv_tab["totalConversionRate"] = pd.to_numeric(conv_tab["totalConversionRate"], errors="coerce")

conv_w_arch = conv_tab.merge(parent_to_arch, on="Parent0", how="left")


**Sales‑sheet helper functions**

This block defines helper utilities to work with the messy sales‑parameter sheet: functions to normalize text, locate header cells, scan header ranges, clean archetype labels, convert year columns and percentages, and ignore non‑data rows.



In [17]:
def norm_txt(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

def find_cell(text):
    target = text.strip().lower()
    for r in range(sales_raw.shape[0]):
        for c in range(sales_raw.shape[1]):
            if target in norm_txt(sales_raw.iat[r, c]):
                return r, c
    return None

def scan_row_until_blank(r, start_c):
    end = start_c
    while (end < sales_raw.shape[1] and
           not pd.isna(sales_raw.iat[r, end]) and
           str(sales_raw.iat[r, end]).strip() != ""):
        end += 1
    return end

def clean_series(s):
    s = s.astype(str).str.strip()
    s = s.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return s

def is_bad_archetype(x):
    t = norm_txt(x)
    if t in ["", "archetype", "maturity"]:
        return True
    bad = [
        "median first year sales volumes",
        "average first year sales volumes",
        "median growth rates",
        "average growth rates",
        "relative sales year",
    ]
    return any(b in t for b in bad)

def normalize_year_cols(cols):
    m = {}
    for c in cols:
        try:
            m[int(float(str(c).strip()))] = c
        except Exception:
            pass
    return m

def to_rate(x):
    if pd.isna(x):
        return 0.0
    if isinstance(x, str):
        s = x.strip()
        if s.endswith("%"):
            s = s[:-1].strip()
            v = pd.to_numeric(s, errors="coerce")
            return 0.0 if pd.isna(v) else float(v) / 100.0
        v = pd.to_numeric(s, errors="coerce")
        if pd.isna(v):
            return 0.0
        v = float(v)
    else:
        v = float(x)
    return v / 100.0 if abs(v) > 2 else v


**Median first‑year sales block**

This block locates the “Median first year sales volumes” section in the sales sheet, extracts a clean Archetype × Maturity table, and builds median_sales_df, which stores baseline first‑year sales volumes for each maturity bucket.



In [18]:
# Locate "Median first year sales volumes" section
anchor = find_cell("Median first year sales volumes")
if anchor is None:
    raise ValueError("Couldn't find 'Median first year sales volumes' block.")
anchor_r, _ = anchor

median_header_r = None
for r in range(anchor_r, min(anchor_r + 30, sales_raw.shape[0])):
    if norm_txt(sales_raw.iat[r, 0]) == "archetype":
        median_header_r = r
        break
if median_header_r is None:
    raise ValueError("Couldn't find 'Archetype' header for Median first year sales block.")

median_start_c = 0
median_end_c = scan_row_until_blank(median_header_r, median_start_c)

median_df = sales_raw.iloc[median_header_r + 1:, median_start_c:median_end_c].copy()
median_df.columns = sales_raw.iloc[median_header_r, median_start_c:median_end_c].values

median_df = median_df.dropna(subset=["Archetype"])
median_df["Archetype"] = clean_series(median_df["Archetype"])
median_df = median_df[~median_df["Archetype"].apply(is_bad_archetype)]

# Map maturity columns
maturity_cols_map = {}
for col in median_df.columns:
    if norm_txt(col) == "archetype":
        continue
    try:
        maturity_cols_map[int(float(str(col).strip()))] = col
    except Exception:
        pass

needed_maturities = [85, 95, 105, 115]
missing = [m for m in needed_maturities if m not in maturity_cols_map]
if missing:
    raise ValueError(
        f"Missing maturity cols {missing} in median sales block. "
        f"Found: {sorted(maturity_cols_map.keys())}"
    )

median_sales_df = median_df[["Archetype"] + [maturity_cols_map[m] for m in needed_maturities]].copy()
median_sales_df.columns = ["Archetype"] + needed_maturities
for m in needed_maturities:
    median_sales_df[m] = pd.to_numeric(median_sales_df[m], errors="coerce")
median_sales_df = median_sales_df.dropna(subset=needed_maturities, how="all")


**Median growth rates block**

This block finds and parses the “Median growth rates” section, cleans Archetype and Maturity, maps generic Year2–Year10 columns, and validates that we have growth‑rate data for all required years used in the sales lifecycle.



In [19]:
growth_anchor = find_cell("Median growth rates")
if growth_anchor is None:
    raise ValueError("Couldn't find 'Median growth rates' block.")
ga_r, _ = growth_anchor

growth_header_r = None
growth_start_c = None
for r in range(ga_r, min(ga_r + 30, sales_raw.shape[0])):
    for c in range(sales_raw.shape[1] - 1):
        if norm_txt(sales_raw.iat[r, c]) == "archetype" and norm_txt(sales_raw.iat[r, c + 1]) == "maturity":
            growth_header_r = r
            growth_start_c = c
            break
    if growth_header_r is not None:
        break
if growth_header_r is None:
    raise ValueError("Couldn't find Archetype+Maturity header in Median growth rates block.")

growth_end_c = scan_row_until_blank(growth_header_r, growth_start_c)

growth_df = sales_raw.iloc[growth_header_r + 1:, growth_start_c:growth_end_c].copy()
growth_df.columns = sales_raw.iloc[growth_header_r, growth_start_c:growth_end_c].values

growth_df = growth_df.dropna(subset=["Archetype", "Maturity"])
growth_df["Archetype"] = clean_series(growth_df["Archetype"])
growth_df["Maturity"]  = pd.to_numeric(growth_df["Maturity"], errors="coerce")
growth_df = growth_df[~growth_df["Archetype"].apply(is_bad_archetype)]
growth_df = growth_df.dropna(subset=["Archetype", "Maturity"])

year_map = normalize_year_cols(
    [c for c in growth_df.columns if norm_txt(c) not in ["archetype", "maturity"]]
)

years_needed = list(range(2, 11))  # Year2..Year10
missing_years = [y for y in years_needed if y not in year_map]
if missing_years:
    raise ValueError(
        f"Missing growth year columns {missing_years} in Median growth rates block. "
        f"Found: {sorted(year_map.keys())}"
    )


**Lookup functions and sales curve**

This block provides:

- `get_median_sales()` and `get_yoy_rates()` — median-based lookups kept as
  **fallback only** in case a salesVariability parameter is missing
- `get_mean_std_by_archetype()` and `sample_yield_conv_normal()` — yield and
  conversion sampling (unchanged)
- `build_sales_curve()` — deterministic median curve (fallback only)
- `draw_sales_curve()` — **new**: draws a stochastic sales curve from the
  lognormal parameters in `salesVariability.xlsx`.  Called once per simulation
  run so each iteration gets its own unique sales trajectory.


In [20]:
def get_median_sales(archetype, maturity):
    """Fallback: return median first-year sales from data.xlsx."""
    row = median_sales_df[median_sales_df["Archetype"] == archetype]
    if row.empty:
        return None
    v = row[maturity].dropna()
    return None if v.empty else float(v.iloc[0])

def get_yoy_rates(archetype, maturity):
    """Fallback: return median YoY growth rates from data.xlsx."""
    row = growth_df[
        (growth_df["Archetype"] == archetype) &
        (growth_df["Maturity"]  == maturity)
    ]
    if row.empty:
        return None
    return [to_rate(row[year_map[y]].iloc[0]) for y in years_needed]

def get_mean_std_by_archetype(archetype):
    """Return mean and std of yield factor and conversion rate for archetype."""
    y = yield_w_arch.loc[yield_w_arch["Archetype"] == archetype, "Yield_Factor"].dropna()
    c = conv_w_arch.loc[conv_w_arch["Archetype"]   == archetype, "totalConversionRate"].dropna()
    y_mean = float(y.mean()) if len(y) else float(yield_w_arch["Yield_Factor"].dropna().mean())
    c_mean = float(c.mean()) if len(c) else float(conv_w_arch["totalConversionRate"].dropna().mean())
    y_std  = float(y.std(ddof=1)) if len(y) > 1 else 1e-6
    c_std  = float(c.std(ddof=1)) if len(c) > 1 else 1e-6
    if y_std <= 0: y_std = 1e-6
    if c_std <= 0: c_std = 1e-6
    return y_mean, y_std, c_mean, c_std

def sample_yield_conv_normal(archetype, rng):
    """Draw one yield factor and one conversion rate for this run."""
    y_mean, y_std, c_mean, c_std = get_mean_std_by_archetype(archetype)
    return (float(max(0.0, rng.normal(y_mean, y_std))),
            float(max(0.0, rng.normal(c_mean, c_std))))

# ── Fallback: deterministic median sales curve ────────────────────────────────
def build_sales_curve(archetype, maturity):
    """
    Returns a fixed 10-year sales array using median values from data.xlsx.
    Used only as a fallback when lognormal parameters are unavailable.
    """
    y1  = get_median_sales(archetype, maturity)
    yoy = get_yoy_rates(archetype, maturity)
    if y1 is None or yoy is None:
        return None
    sales = [y1]
    for rate in yoy:
        sales.append(max(0.0, sales[-1] * (1 + rate)))
    return np.array(sales, dtype=float)

# ── Primary: stochastic lognormal sales curve (called once per MC run) ────────
def draw_sales_curve(archetype, maturity, rng):
    """
    Draw one complete 10-year sales trajectory from lognormal parameters.

    Year 1 is sampled from Lognormal(mu, sigma²) for first-year sales.
    Each subsequent year's value is computed by multiplying the prior year
    by a growth multiplier drawn from Lognormal(mu, sigma²) for that year.

    Growth rate parameters near -9.21 (= log(~0)) signal end-of-lifecycle —
    the drawn multiplier will be near zero, naturally collapsing sales.

    Falls back to build_sales_curve() if any parameter is missing.
    """
    fy_mu   = sv_fy_mu.get((archetype, maturity))
    fy_sig2 = sv_fy_sig2.get((archetype, maturity))

    if fy_mu is None or fy_sig2 is None:
        # Fallback to median curve if lognormal params unavailable
        return build_sales_curve(archetype, maturity)

    # Draw Year 1 sales from Lognormal(mu, sigma²)
    # numpy lognormal takes (mu, sigma) not sigma²
    fy_sigma = float(np.sqrt(max(fy_sig2, 1e-10)))
    y1 = float(rng.lognormal(fy_mu, fy_sigma))

    sales = [y1]
    for yr in range(2, 11):   # Years 2 through 10
        gr_mu   = sv_gr_mu.get((archetype, maturity, yr))
        gr_sig2 = sv_gr_sig2.get((archetype, maturity, yr))

        if gr_mu is None or gr_sig2 is None:
            # Fallback: apply median growth rate for this year
            yoy = get_yoy_rates(archetype, maturity)
            if yoy is not None:
                rate = yoy[yr - 2]
                sales.append(max(0.0, sales[-1] * (1.0 + rate)))
            else:
                sales.append(0.0)
            continue

        # Draw growth multiplier from Lognormal(mu, sigma²)
        # exp(draw) gives the multiplicative factor for this year
        gr_sigma     = float(np.sqrt(max(gr_sig2, 1e-10)))
        growth_draw  = float(rng.lognormal(gr_mu, gr_sigma))
        next_sales   = sales[-1] * growth_draw
        sales.append(max(0.0, next_sales))

    return np.array(sales, dtype=float)


**Widgets (archetype, maturity, strategy, multipliers, rules, simulation)**

This block defines the interactive controls for the dashboard: a multi‑select list of products (Archetype | Maturity), production strategy and per‑year multipliers, simulation settings (iterations and seed), analysis‑year and threshold selectors, and knobs for carryover stop‑rules and minimum production floors.



In [21]:
# ==== PRODUCT SELECTORS (multi‑product) ====

# Build list of valid Archetype | Maturity combinations
product_options = []
for a in sorted(median_sales_df["Archetype"].dropna().unique()):
    for m in [85, 95, 105, 115]:
        y1 = get_median_sales(a, m)
        if y1 is not None and not np.isnan(y1):
            product_options.append(f"{a} | {m}")

products_ms = widgets.SelectMultiple(
    options=product_options,
    value=(product_options[0],),
    description="Products:",
    layout=widgets.Layout(width="600px", height="140px")
)


# ==== PRODUCTION STRATEGY ====
strategy_dd = widgets.Dropdown(
    options=[
        ("Custom multiplier by year (Y1–Y10)", "custom"),
        ("Just‑in‑time 1.0×", "jit"),
        ("Conservative 1.2×", "cons"),
        ("Aggressive 2.0×", "aggr"),
    ],
    value="custom",
    description="Strategy:",
    layout=widgets.Layout(width="480px")
)

# 10 separate sliders, one for each year
year_mult_sliders = [
    widgets.FloatSlider(
        description=f"Y{i}",
        min=0.5, max=3.0, step=0.1, value=1.5,
        layout=widgets.Layout(width="260px")
    )
    for i in range(1, 11)
]

def get_yearly_multipliers():
    strat = strategy_dd.value
    if strat == "custom":
        return [float(s.value) for s in year_mult_sliders]  # Y1..Y10
    elif strat == "jit":
        return [1.0] * 10
    elif strat == "cons":
        return [1.2] * 10
    elif strat == "aggr":
        return [2.0] * 10
    else:
        return [1.5] * 10


# ==== SIMULATION SETTINGS ====
iters_slider = widgets.IntSlider(
    description="Simulations:",
    min=100, max=5000, step=100, value=1000,
    layout=widgets.Layout(width="350px")
)
seed_box = widgets.IntText(
    value=42,
    description="Random seed:",
    layout=widgets.Layout(width="220px")
)

# ==== ANALYSIS YEAR + THRESHOLD (v5-style) ====
year_mode_dd = widgets.Dropdown(
    options=[
        ("Last Year of Sales", "last_sales"),
        ("Choose specific year", "custom")
    ],
    value="last_sales",
    description="Year mode:",
    layout=widgets.Layout(width="350px")
)
analysis_year_dd = widgets.Dropdown(
    options=[(f"Year {i}", i-1) for i in range(1, 11)],
    value=5-1,
    description="Analysis year:",
    layout=widgets.Layout(width="250px")
)
threshold_slider = widgets.FloatSlider(
    description="Remaining threshold:",
    min=-10000, max=50000, step=500, value=0.0,
    layout=widgets.Layout(width="500px")
)

# ==== PRODUCTION CONSTRAINTS: CARRYOVER RULE + MIN FLOOR ====
carryover_k_slider = widgets.FloatSlider(
    description="Stop if carryover > k× next-yr sales (k):",
    min=0.0, max=5.0, step=0.1, value=0.0,
    layout=widgets.Layout(width="600px")
)

enable_min_floor_chk = widgets.Checkbox(
    value=False,
    description="Enable minimum production floor",
    indent=False
)
min_floor_slider = widgets.FloatSlider(
    description="Min production floor:",
    min=0.0, max=20000, step=500, value=0.0,
    layout=widgets.Layout(width="500px")
)

**Strategy multipliers and one-run simulation**

This block defines:

- `get_yearly_multipliers()` — reads the 10 per-year multiplier sliders
  (or returns fixed values for named strategies)
- `simulate_one_run(sales, archetype, rng)` — runs one complete 10-year
  lifecycle given a sales array, archetype, and random generator.
  Yield and conversion are sampled once per run from Normal(mean, std).
- `_determine_analysis_year()` — resolves the Year Mode widget to a 0-based
  year index for probability stats
- `build_lifecycle_sim()` — Monte Carlo runner that now calls
  `draw_sales_curve()` inside the iteration loop so each run gets its own
  stochastic sales trajectory drawn from lognormal parameters.
  Yield and conversion variability is also applied per run as before.


In [22]:
def get_yearly_multipliers():
    strat = strategy_dd.value
    if strat == "custom":
        return [float(s.value) for s in year_mult_sliders]
    elif strat == "jit":
        return [1.0] * 10
    elif strat == "cons":
        return [1.2] * 10
    elif strat == "aggr":
        return [2.0] * 10
    else:
        return [1.5] * 10  # fallback

def simulate_one_run(sales, archetype, rng):
    """
    One Monte Carlo lifecycle:
    - sales: length-10 array (Year 1..10)
    - global widgets: get_yearly_multipliers(), carryover_k_slider,
      enable_min_floor_chk, min_floor_slider
    """
    # One draw of yield + conversion per run (can be moved inside loop if needed)
    y_draw, c_draw = sample_yield_conv_normal(archetype, rng)

    carryover = 0.0  # Year 1 carryover must be 0
    rows = []
    missed_sales = []

    mults = get_yearly_multipliers()
    k = float(carryover_k_slider.value)
    use_floor = bool(enable_min_floor_chk.value)
    min_floor = float(min_floor_slider.value)

    for yr in range(10):
        expected_sales_next = sales[yr+1] if yr < 9 else 0.0

        # Strategy-based planned production
        planned_prod = mults[yr] * expected_sales_next

        # Carryover-based stop rule (if k > 0)
        if k > 0 and expected_sales_next > 0 and carryover > k * expected_sales_next:
            planned_prod = 0.0

        # Minimum production floor (if enabled and we're producing)
        if use_floor and planned_prod > 0.0:
            planned_prod = max(planned_prod, min_floor)

        new_prod = planned_prod * y_draw * c_draw

        prod_loss  = new_prod * 0.02
        carry_loss = carryover * 0.10

        total_saleable = (carryover - carry_loss) + (new_prod - prod_loss)
        remaining      = total_saleable - sales[yr]  # negative allowed

        missed = max(0.0, -remaining)
        missed_sales.append(missed)

        rows.append([
            carryover,
            -carry_loss,
            planned_prod,
            new_prod,
            -prod_loss,
            total_saleable,
            sales[yr],
            remaining
        ])
        carryover = remaining

    return np.array(rows, dtype=float), np.array(missed_sales, dtype=float)


def _determine_analysis_year(lifecycle_df):
    """
    Implements v5 'Year mode' logic.
    """
    mode = year_mode_dd.value
    sales_row = lifecycle_df.loc["Sales"].astype(float).values
    if mode == "last_sales":
        # Last year with positive mean sales; fallback to Year 10 if all zero
        idx = np.where(sales_row > 0)[0]
        return int(idx[-1]) if len(idx) else 9
    else:
        return int(analysis_year_dd.value)

def build_lifecycle_sim(products, iterations, seed):
    """
    products: tuple of strings like 'Archetype | 85' from products_ms
    """
    # Parse selections and build sales curves
    # Parse selected product strings and validate each has sales data
    parsed = []
    for p in products:
        arch, mat_str = p.split("|")
        arch     = arch.strip()
        maturity = int(mat_str.strip())
        # Validate using fallback median curve (just checks data exists)
        if build_sales_curve(arch, maturity) is None:
            print(f"\u274c Missing sales data for {arch} | {maturity}; skipping.")
            continue
        parsed.append((arch, maturity))

    if not parsed:
        print("\u274c No valid products selected.")
        return

    rng = np.random.default_rng(int(seed))
    run_rows_all = []  # list of (iters, 10, 8) arrays, one per product

    # Run Monte Carlo separately for each product
    # draw_sales_curve() is called INSIDE the loop — each run gets its own
    # stochastic sales trajectory drawn from lognormal parameters
    for arch, maturity in parsed:
        prod_rows = []
        for _ in range(int(iterations)):
            sales = draw_sales_curve(arch, maturity, rng)   # unique per run
            rows, _ = simulate_one_run(sales, arch, rng)
            prod_rows.append(rows)
        prod_rows = np.stack(prod_rows, axis=0)  # (iters, 10, 8)
        run_rows_all.append(prod_rows)

    # Aggregate across products: sum inventory / production / sales
    run_rows = np.sum(run_rows_all, axis=0)      # (iters, 10, 8)

    mean_rows = run_rows.mean(axis=0)            # (10, 8)
    cols = [f"Year {i}" for i in range(1, 11)]
    lifecycle_df = pd.DataFrame(
        mean_rows.T,
        columns=cols,
        index=[
            "Carry-in inventory (from prior year)",
            "Carry-in quality loss (10%)",
            "Planned production",
            "Actual production (after yield & conversion)",
            "Production quality loss (2%)",
            "Total saleable inventory",
            "Sales",
            "Remaining inventory [negative = stockout]"
        ]
    )

    # Remaining inventory per run (aggregated) for probabilities
    remaining_all = run_rows[:, :, 7]

    # Determine analysis year index (v5 logic) and threshold
    ay_idx = _determine_analysis_year(lifecycle_df)
    thr = float(threshold_slider.value)

    sel_rem = remaining_all[:, ay_idx]
    end_rem = remaining_all[:, 9]

    def _row_stats(arr):
        return {
            "Mean remaining":   float(arr.mean()),
            "Median remaining": float(np.median(arr)),
            "P90 remaining":    float(np.percentile(arr, 90)),
            "P(remaining > 0)": float((arr > 0).mean()),
            f"P(remaining > {thr:.0f})": float((arr > thr).mean()),
            "P(stockout) — below zero": float((arr < 0).mean())
        }

    selected_stats = _row_stats(sel_rem)
    end_stats      = _row_stats(end_rem)

    summary_df = pd.DataFrame.from_dict(
        {
            f"Selected year (Year {ay_idx+1})": selected_stats,
            "End of lifecycle (Year 10)":       end_stats
        },
        orient="index"
    )

    product_list_str = "; ".join([f"{a}|{m}" for a, m in parsed])

    print("====================================================")
    print(f"v5 Monte Carlo   |   {product_list_str}")
    print(f"Strategy  : {strategy_dd.label}")
    print("Distribution: Lognormal (from salesVariability.xlsx)   |   Yield & conversion: Normal")
    print(f"Simulations: {iterations:,}   |   Random seed: {seed}")
    print(f"Analysis year : Year {ay_idx+1}  (mode: "
          f"{'Last Year of Sales' if year_mode_dd.value=='last_sales' else 'Custom'})")
    print("====================================================\n")

    print("Mean lifecycle across all simulations (aggregated across products):\n")
    display(lifecycle_df.round(1))

    print(f"\nInventory probability summary  (threshold = {thr:.0f} units):\n")
    display(summary_df.round(3))


**Portfolio simulation and inventory summary - Interactive wiring and layout**

This block also aggregates the simulation across all selected products, averaging and summing their lifecycles, then computes the inventory‑risk summary metrics (mean, median, P90, and probabilities above zero and above the user‑defined remaining‑inventory threshold) for both the selected analysis year and end of lifecycle.

This block wires the widgets into a single interactive dashboard: when the user clicks “Run simulation”, the current control values are captured, the Monte Carlo engine is executed, and the aggregated lifecycle table plus inventory probability summary are rendered directly below the controls.




In [23]:
# Button to apply parameters
run_button = widgets.Button(
    description="Run simulation",
    button_style="primary",
    layout=widgets.Layout(width="200px")
)

# This dict will hold the "frozen" parameters when you click the button
_current_params = {"products": None, "iterations": None, "seed": None}

def on_run_clicked(b):
    # Capture current widget values into the params dict
    _current_params["products"] = tuple(products_ms.value)
    _current_params["iterations"] = int(iters_slider.value)
    _current_params["seed"] = int(seed_box.value)

    # Update selection label
    n = len(_current_params["products"])
    selection_label.value = f"Selected products: <b>{n}</b>"

    # Clear and call the simulation manually
    output_area.clear_output()
    with output_area:
        build_lifecycle_sim(
            _current_params["products"],
            _current_params["iterations"],
            _current_params["seed"],
        )


run_button.on_click(on_run_clicked)

# Output area where tables/summary will appear
output_area = widgets.Output()

selection_label = widgets.HTML(
    value="Selected products: 0",
)


# Full dashboard layout (controls + button + output together)
dashboard = widgets.VBox([
    widgets.HTML("<h3>v5 Monte Carlo Dashboard</h3>"),
    widgets.HTML("<b>Product(s)</b>"),
    products_ms,
    selection_label,
    widgets.HTML("<hr>"),
    widgets.HTML("<b>Production strategy</b> "
                 "<small>(how many units to plan relative to next year's expected sales)</small>"),
    strategy_dd,
    widgets.VBox(year_mult_sliders),
    widgets.HTML("<hr>"),
    widgets.HTML("<b>Simulation settings</b>"),
    widgets.HBox([iters_slider, seed_box]),
    widgets.HTML("<hr>"),
    widgets.HTML("<b>Analysis year</b> "
                 "<small>(which year to show detailed probability stats for)</small>"),
    widgets.HBox([year_mode_dd, analysis_year_dd]),
    widgets.HTML("<b>Carryover threshold</b> "
                 "<small>(units — used to compute P(remaining > threshold))</small>"),
    threshold_slider,
    widgets.HTML("<hr>"),
    widgets.HTML("<b>Production constraints</b> "
                 "<small>(minimum floor + optional carryover rule)</small>"),
    widgets.HBox([enable_min_floor_chk]),
    min_floor_slider,
    carryover_k_slider,
    widgets.HTML("<hr>"),
    run_button,
    widgets.HTML("<br>"),
    output_area,
])

display(dashboard)
